In [8]:
from utils.data_helpers import initialize_metadata_data, initialize_stock_data

await initialize_stock_data()
await initialize_metadata_data()

2025-11-06 22:56:42,223 - utils.data_helpers - INFO - Stock data already initialized, skipping
2025-11-06 22:56:42,223 - utils.data_helpers - INFO - Metadata already initialized, skipping


In [ ]:
# Get the initialized dataframes
from utils.data_helpers import get_metadata_dataframe, get_stocks_dataframe

stocks_df = get_stocks_dataframe().copy()
metadata_df = get_metadata_dataframe().reset_index()

# Count total files per fincode
total_files_per_fincode = metadata_df.groupby("fincode").size().rename("total_files")

# Count files per subcatname per fincode (pivot table)
subcatname_counts = (
    metadata_df.groupby(["fincode", "subcatname"]).size().unstack(fill_value=0)
)
# Add prefix to subcatname columns for clarity
subcatname_counts.columns = [f"count_{col}" for col in subcatname_counts.columns]

# Merge total counts into stocks dataframe
enriched_df = stocks_df.join(total_files_per_fincode, how="left")

# Merge subcatname counts
enriched_df = enriched_df.join(subcatname_counts, how="left")

# Fill NaN values with 0 for stocks that have no metadata files
enriched_df = enriched_df.fillna(0)

# Convert count columns to integers
count_columns = ["total_files"] + [
    col for col in enriched_df.columns if col.startswith("count_")
]
enriched_df[count_columns] = enriched_df[count_columns].astype(int)

# Save to CSV
enriched_df.to_csv("active_equity_and_sub_listing_with_metadata_counts.csv")


Enriched DataFrame shape: (7481, 10)
Total stocks: 7481
Stocks with metadata: 4001

Subcategories found: 3

Sample of enriched data:
|   fincode |   dbId |   scripcode | compname                          |   indCode | symbol   | bseScripId   |   total_files |   count_annual-report |   count_concall |   count_investor-presentation |
|----------:|-------:|------------:|:----------------------------------|----------:|:---------|:-------------|--------------:|----------------------:|----------------:|------------------------------:|
|    100002 |      1 |      500002 | ABB India Ltd.                    |        39 | ABB      | ABB          |             2 |                     1 |               1 |                             0 |
|    100003 |      2 |      500003 | Aegis Logistics Ltd.              |       104 | AEGISLOG | AEGISLOG     |            16 |                     1 |               4 |                            11 |
|    100008 |      4 |      500008 | Amara Raja Energy & Mobil